In [5]:
!pip install timm

  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.3.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.6 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.6 MB 1.1 MB/s eta 0:00:02
   ---------------- ----------------------- 1.0/2.6 MB 1.6 MB/s eta 0:00:01
   ---------------- ----------------------- 1.0/2.6 MB 1.6 MB/s eta 0:00:01
   -------------------- ------------------- 1.3/2.6 MB 1.3 MB/s eta 0:00:01
   ------

In [7]:
!pip uninstall timm -y

Found existing installation: timm 1.0.26
Uninstalling timm-1.0.26:
  Successfully uninstalled timm-1.0.26


In [8]:
!pip install timm==0.6.13

  Using cached timm-0.6.13-py3-none-any.whl.metadata (38 kB)
Using cached timm-0.6.13-py3-none-any.whl (549 kB)


In [9]:
import torch
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

from mtcnn import MTCNN
from torchvision import transforms

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using Device:", device)

Using Device: cuda


In [11]:
MODEL_PATH = "Models/FFPP_model_20_epochs_99acc.pt"

print("🔄 Loading model...")

checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)

model = checkpoint['model']
model.load_state_dict(checkpoint['model_state_dict'])

model.to(device)
model.eval()

print("✅ Model Loaded Successfully")

🔄 Loading model...
✅ Model Loaded Successfully


In [12]:
print("🔄 Initializing MTCNN...")
detector = MTCNN()
print("✅ MTCNN ready")

🔄 Initializing MTCNN...
✅ MTCNN ready


In [13]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

print("✅ Transform matched with training (128x128)")

✅ Transform matched with training (128x128)


In [14]:
def get_face(frame):
    try:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        detections = detector.detect_faces(rgb)

        if len(detections) == 0:
            print("⚠️ No face detected")
            return None

        # Take best face
        detections = sorted(detections, key=lambda x: x['confidence'], reverse=True)
        x, y, w, h = detections[0]['box']

        x = max(0, x)
        y = max(0, y)

        face = frame[y:y+h, x:x+w]

        if face.size == 0:
            return None

        return face

    except Exception as e:
        print("❌ Face detection error:", e)
        return None

In [15]:
def predict_video(video_path, frame_skip=20):
    try:
        print("\n🎥 Processing video:", video_path)

        if not os.path.exists(video_path):
            raise FileNotFoundError("Video not found!")

        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            raise ValueError("Cannot open video!")

        frame_id = 0
        processed_frames = 0
        fake_frames = []
        predictions = []

        print("🔄 Starting frame extraction...")

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_id % frame_skip == 0:
                print(f"\n📸 Frame {frame_id}")

                face = get_face(frame)

                if face is None:
                    print("⏭️ Skipping (no face)")
                    frame_id += 1
                    continue

                try:
                    face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
                    face_tensor = transform(face_rgb).unsqueeze(0).to(device)

                    print("⚙️ Running model...")

                    with torch.no_grad():
                        output = model(face_tensor)
                        prob = torch.sigmoid(output).item()

                    pred = 1 if prob > 0.5 else 0
                    label = "FAKE" if pred == 1 else "REAL"

                    print(f"📊 {label} ({prob:.4f})")

                    processed_frames += 1
                    predictions.append(pred)

                    if pred == 1:
                        fake_frames.append((face_rgb, prob))

                except Exception as e:
                    print("❌ Frame error:", e)

            frame_id += 1

        cap.release()

        # -------------------------
        # FINAL DECISION
        # -------------------------
        if processed_frames == 0:
            print("❌ No valid frames processed")
            return None

        fake_ratio = sum(predictions) / processed_frames
        final_label = "FAKE" if fake_ratio > 0.3 else "REAL"

        print("\n📊 FINAL RESULT")
        print("🎯 Video Prediction:", final_label)
        print(f"📈 Fake Frames: {sum(predictions)}/{processed_frames}")
        print(f"📊 Fake Ratio: {fake_ratio:.2f}")

        # -------------------------
        # SHOW TOP FAKE FRAMES
        # -------------------------
        if len(fake_frames) > 0:
            print("\n🖼️ Top Fake Frames:")

            fake_frames = sorted(fake_frames, key=lambda x: x[1], reverse=True)

            for i, (img, prob) in enumerate(fake_frames[:3]):
                plt.imshow(img)
                plt.title(f"Fake Frame {i+1} | Confidence: {prob:.4f}")
                plt.axis("off")
                plt.show()
        else:
            print("✅ No fake frames detected")

        return final_label

    except Exception as e:
        print("❌ Error:", e)
        return None

In [18]:
video_path = "E:/CSMSS Projects/Sem6_mini_project/FaceForensics++_C23/Face2Face/027_009.mp4"   # 🔁 change this

predict_video(video_path, frame_skip=5)


🎥 Processing video: E:/CSMSS Projects/Sem6_mini_project/FaceForensics++_C23/Face2Face/027_009.mp4
🔄 Starting frame extraction...

📸 Frame 0
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 5
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 10
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 15
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 20
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 25
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 30
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 35
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 40
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 45
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 50
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 55
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 60
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 65
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 70
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 75
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 80
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame 85
⚙️ Running model...
📊 REAL (0.0000)

📸 Frame

'REAL'